# Capstone pipeline

Full Capstone project requires:
1. Download DeepFashion images to train recognition of cloth type and condition.
2. Download Poshmark and Depop sample data for a few cloth types: price, brand, type, condition
3. Clean (remove missing) and unify (rename attributes to be the same across stores) data
4. Prepare data (extract features, prepare training, validation and testing datasets, as well as create synthetic data to similate conditions)
5. Train (run visual training, run categorical training)

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Finding where the notebook is located. The path is used for some more complicated script calls instead of making this notebook overgrown.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'README.md').exists() and (PROJECT_ROOT.parent / 'README.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

sys.path.append('scripts')
sys.path.append('src')
PYTHON = sys.executable

def run(args, check=True, env=None):
    if isinstance(args, str):
        print(f"$ {args}")
        return subprocess.run(args, shell=True, check=check, env=env)
    cmd = ' '.join(shlex.quote(str(part)) for part in args)
    print(f"$ {cmd}")
    return subprocess.run([str(part) for part in args], check=check, env=env)

print(PROJECT_ROOT)
print(PYTHON)


C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone
C:\ProgramData\anaconda3\python.exe


In [2]:
KAGGLE_DATASETS = [
    ('thusharanair/deepfashion2-original-with-dataframes', 'data/raw/deepfashion/kaggle_thushan'),
    ('vishalbsadanand/deepfashion-1', 'data/raw/deepfashion/kaggle_vishal'),
]

SCRAPE_JOBS = [
    {'query': 'dress', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'pants', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'shorts', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'jeans', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'skirt', 'max_items': 500, 'rate_limit': 2.0},
]

SYNTHETIC_INPUT_DIR = 'data/deepfashion/original'
SYNTHETIC_OUTPUT_DIR = 'data/deepfashion/synthetic_degraded'
SYNTHETIC_VARIATIONS = 5
SYNTHETIC_MAX_IMAGES = 5000

CLOTHING_TYPE_DIR = 'data/processed/clothing_type'
CONDITION_DIR = 'data/processed/condition_assessment'
MULTITASK_DIR = 'data/processed/multitask'
FEATURES_DIR = 'data/features'
EMBEDDINGS_DIR = 'data/embeddings'
PRICE_DATASET_DIR = 'data/price_classification'
VECTOR_INDEX_DIR = 'data/vector_index'

UNIFIED_DATA_CSV = 'data/scraped/unified/unified_data.csv'

CLOTHING_TYPE_MODEL_DIR = 'models/clothing_type'
CONDITION_MODEL_DIR = 'models/condition_assessment'
MULTITASK_MODEL_DIR = 'models/multitask'
PRICE_MODEL_DIR = 'models/price_model'

VISION_MODEL_FOR_FEATURES = f'{MULTITASK_MODEL_DIR}/checkpoints/best_loss.pth'

TRAIN_DEVICE = 'cuda'

In [3]:
#run(["pip", "install", "gdown"])
#run(["pip", "install", "playwright"])
#run([PYTHON, '-m', 'playwright', 'install', 'chromium'])

## Warning! Next operation will download ~56GBs of images

## Download

In [3]:
import download_deepfashion
import download_kaggle_datasets
import process_deepfashion_to_csv
import generate_statistics

#download_deepfashion.main(['--subset', 'all', '--download', '--extract', '--verify']) # this can only be used with signed in google account. commenting it out for public repo.
# Don't forget to set KAGGLE_USERNAME and KAGGLE_KEY in system environment variables
#for dataset, output_dir in KAGGLE_DATASETS:
    #download_kaggle_datasets.main(['--dataset', dataset, '--output', output_dir])

process_deepfashion_to_csv.main(['--input', 'data/raw/deepfashion', '--output', 'data/processed'])
generate_statistics.main(['--processed-dir', 'data/processed', '--output', 'data/processed/deepfashion_statistics.csv'])

2026-03-25 22:59:06,466 - download_kaggle_datasets - ERROR - Kaggle credentials not found!
2026-03-25 22:59:06,467 - download_kaggle_datasets - ERROR - Please set KAGGLE_USERNAME and KAGGLE_KEY environment variables
2026-03-25 22:59:06,467 - download_kaggle_datasets - ERROR - Or configure them in your .env file


SystemExit: 1

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Scrape

In [ ]:
import scrape_marketplaces

for job in SCRAPE_JOBS:
    scrape_marketplaces.main(['--query', job['query'], '--all-platforms', '--max-items', str(job['max_items']), '--rate-limit', str(job['rate_limit']), '--output-dir', 'data/raw/scraped'])

2026-03-25 22:27:27,285 - PoshmarkScraper - INFO - Starting Poshmark search scrape: 'dress' (max: 500)
2026-03-25 22:27:27,286 - PoshmarkScraper - INFO - Scraping page 1...


Starting scrape: POSHMARK
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:27:28,500 - PoshmarkScraper - INFO - Playwright browser initialized
2026-03-25 22:27:35,552 - PoshmarkScraper - WARNING - Timeout waiting for selector: [class*="tile"], [class*="card"], [class*="item"]
2026-03-25 22:27:35,791 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:41,489 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:41,491 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:47,028 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:47,029 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:52,495 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:52,496 - PoshmarkScraper - INFO - S


 Failed to scrape poshmark: 'PoshmarkScraper' object has no attribute '_browser_thread'

Starting scrape: THREDUP
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:53:52,756 - ThredUpScraper - INFO - Playwright browser initialized
2026-03-25 22:53:52,809 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=1
2026-03-25 22:53:53,599 - ThredUpScraper - INFO - Status=200 Title=Used Dress On Sale Up To 90% Off | ThredUp
2026-03-25 22:53:58,166 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-25 22:53:58,443 - ThredUpScraper - INFO - Scraping page 2...
2026-03-25 22:53:58,481 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=2
2026-03-25 22:53:58,578 - ThredUpScraper - INFO - Status=403 Title=Just a moment...
2026-03-25 22:54:01,660 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-25 22:54:01,666 - ThredUpScraper - INFO - No more items found
2026-03-25 22:54:01,666 - ThredUpScraper - INFO - Completed scraping 0 items from ThredUp
2026-03-25 22:54:01,667 - DepopScraper


 Failed to scrape thredup: 'ThredUpScraper' object has no attribute '_browser_thread'

Starting scrape: DEPOP
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:54:02,207 - DepopScraper - INFO - Playwright browser initialized with anti-bot detection avoidance
2026-03-25 22:54:02,430 - DepopScraper - WARNING - API scraping failed, falling back to web scraping
2026-03-25 22:54:04,284 - DepopScraper - INFO - Waiting for products to load (10 seconds)...
2026-03-25 22:54:16,328 - DepopScraper - INFO - Found 24 product links
2026-03-25 22:54:16,334 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/brynbnoz-princess-polly-white-strapless-mini-2976/
2026-03-25 22:54:25,673 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/chvoechea57-2x-dress-fits-like-large-e4e7/
2026-03-25 22:54:35,058 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/blingraham813-cider-brown-multi-coloured-long-4b71/
2026-03-25 22:54:44,373 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/cadencehmlt-altard-state-brown-floral-dress-d6db/
2026-03-25 22:54:53,785 - DepopScraper - INFO - Sc


 Failed to scrape depop: 'DepopScraper' object has no attribute '_browser_thread'

SCRAPING COMPLETE
Total items scraped: 0
  poshmark: 0 items
  thredup: 0 items
  depop: 0 items

Data saved to: data/raw/scraped
Starting scrape: POSHMARK
Query: pants
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:58:01,270 - PoshmarkScraper - INFO - Playwright browser initialized
2026-03-25 22:58:05,751 - PoshmarkScraper - WARNING - Timeout waiting for selector: [class*="tile"], [class*="card"], [class*="item"]
2026-03-25 22:58:05,995 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:11,474 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:11,475 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:16,933 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:16,934 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:22,428 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:22,430 - PoshmarkScraper - INFO - Scraping item: https://poshmark

## Clean And Organize

In [ ]:
run([
    PYTHON,
    'scripts/clean_scraped_data.py',
    '--input-dir', 'data/raw/scraped',
    '--output-dir', 'data/scraped',
    '--remove-no-images',
])

run([PYTHON, 'scripts/organize_data.py', '--create-dirs'])

organize_cmd = [PYTHON, 'scripts/organize_data.py', '--organize-deepfashion', '--copy']
run(organize_cmd)

run([PYTHON, 'scripts/organize_data.py', '--verify'])
run([PYTHON, 'scripts/organize_data.py', '--unify', '--format', 'both'])
run([
    PYTHON,
    'scripts/organize_data.py',
    '--summary',
    '--output', 'data/data_organization_report.json',
])


## Prepare

In [ ]:
synthetic_cmd = [
    PYTHON,
    'scripts/generate_synthetic_data.py',
    '--input-dir', SYNTHETIC_INPUT_DIR,
    '--output-dir', SYNTHETIC_OUTPUT_DIR,
    '--num-variations', str(SYNTHETIC_VARIATIONS),
    '--save-metadata',
    '--max-images', str(SYNTHETIC_MAX_IMAGES),
]
run(synthetic_cmd)

run([
    PYTHON,
    'scripts/prepare_splits.py',
    '--all',
    '--clean',
    '--stratify', 'condition',
    '--report', 'data/processed/split_report.json',
])

run([
    PYTHON,
    'scripts/prepare_clothing_type_dataset.py',
    '--deepfashion-root', 'data/deepfashion',
    '--output-dir', CLOTHING_TYPE_DIR,
])

run([
    PYTHON,
    'scripts/prepare_condition_dataset.py',
    '--deepfashion-dir', 'data/deepfashion/original',
    '--synthetic-dir', SYNTHETIC_OUTPUT_DIR,
    '--output-dir', CONDITION_DIR,
])

run([
    PYTHON,
    'scripts/prepare_multitask_dataset.py',
    '--deepfashion2-csv', 'data/processed/deepfashion2_processed.csv',
    '--synthetic-root', SYNTHETIC_OUTPUT_DIR,
    '--output-dir', MULTITASK_DIR,
])


## Train Vision Models

In [ ]:
clothing_type_cmd = [
    PYTHON,
    'scripts/train_clothing_type.py',
    '--data-dir', CLOTHING_TYPE_DIR,
    '--output-dir', CLOTHING_TYPE_MODEL_DIR,
    '--device', TRAIN_DEVICE,
]
run(clothing_type_cmd)

condition_cmd = [
    PYTHON,
    'scripts/train_condition_assessment.py',
    '--train-csv', f'{CONDITION_DIR}/train.csv',
    '--val-csv', f'{CONDITION_DIR}/val.csv',
    '--data-dir', str(PROJECT_ROOT),
    '--output-dir', CONDITION_MODEL_DIR,
]
run(condition_cmd)

multitask_cmd = [
    PYTHON,
    'scripts/train_multitask_model.py',
    '--train-csv', f'{MULTITASK_DIR}/train.csv',
    '--val-csv', f'{MULTITASK_DIR}/val.csv',
    '--output-dir', MULTITASK_MODEL_DIR,
]
run(multitask_cmd)


## Train Price Model

In [ ]:
extract_cmd = [
    PYTHON,
    'scripts/extract_features.py',
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', FEATURES_DIR,
    '--batch-size', '32',
    '--num-workers', '0',
    '--device', TRAIN_DEVICE,
]
if Path(VISION_MODEL_FOR_FEATURES).exists():
    extract_cmd += ['--vision-model', VISION_MODEL_FOR_FEATURES]
else:
    extract_cmd.append('--no-vision')
run(extract_cmd)

run([
    PYTHON,
    'scripts/prepare_price_dataset.py',
    '--features-dir', FEATURES_DIR,
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', PRICE_DATASET_DIR,
    '--strategy', 'quantile',
    '--n-bins', '5',
])

price_cmd = [
    PYTHON,
    'scripts/train_price_model.py',
    '--features-dir', PRICE_DATASET_DIR,
    '--model', 'xgboost',
    '--output-dir', PRICE_MODEL_DIR,
    '--verbose',
]
run(price_cmd)


## Generate Embeddings

In [ ]:
embedding_mode = 'multimodal' if Path(VISION_MODEL_FOR_FEATURES).exists() else 'text'
embedding_cmd = [
    PYTHON,
    'scripts/generate_embeddings.py',
    '--data-csv', UNIFIED_DATA_CSV,
    '--output-dir', EMBEDDINGS_DIR,
    '--mode', embedding_mode,
    '--batch-size', '32',
    '--device', TRAIN_DEVICE,
]
if Path(VISION_MODEL_FOR_FEATURES).exists():
    embedding_cmd += ['--vision-model', VISION_MODEL_FOR_FEATURES]
run(embedding_cmd)

## Build Vector Index

In [ ]:
index_cmd = [
    PYTHON,
    'scripts/build_vector_index.py',
    '--embeddings-dir', EMBEDDINGS_DIR,
    '--output-dir', VECTOR_INDEX_DIR,
    '--output-name', 'clothing_index',
]
run(index_cmd)
